# Setup pyspark

In [ ]:
!pip uninstall -y pyspark
!pip install pyspark==3.5.1

Found existing installation: pyspark 4.0.3
Uninstalling pyspark-4.0.3:
  Successfully uninstalled pyspark-4.0.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=1c4dc0614484129d230160689f9d5e4acdbbec982b9dde9f20fee9ba67582d82
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, 

# prepare Spark environment in Colab and Bronze Layer

In [ ]:
import os
import time
from pyspark.sql import SparkSession

# Initialize Spark Session
scala_version = '2.12'
spark_version = '3.5.1'
packages = f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}'

spark = SparkSession.builder \
    .appName("SmartGrid_Pipeline") \
    .config("spark.jars.packages", packages) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Kafka Configuration
BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
API_KEY = "SKIUYYELSQrtrfdsf"
API_SECRET = "cfltLdHPgkUVVBKUuf5nF4aCTH8vF+Ft1yhVjjvk0JwerthjgdfsaASDWSbcCg"
TOPIC_NAME = "smart-grid-city-data"

jaas_config = f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'

# Read Stream
raw_stream_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS) \
    .option("subscribe", TOPIC_NAME) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", jaas_config) \
    .option("startingOffsets", "earliest") \
    .load()

string_stream_df = raw_stream_df.selectExpr("CAST(value AS STRING) as json_payload", "timestamp as kafka_time")

# Write to memory
query = string_stream_df \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("live_bronze_table") \
    .start()

print("Bronze Layer: Spark is actively listening to Confluent Cloud...")

Bronze Layer: Spark is actively listening to Confluent Cloud...


# **silver layer**

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col, from_json, to_timestamp, concat_ws

# 1. Exact Schema mapping the actual JSON payload
json_schema = StructType([
    StructField("meter_id", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Time", StringType(), True),
    StructField("Global_active_power", DoubleType(), True),
    StructField("Global_reactive_power", DoubleType(), True),
    StructField("Voltage", DoubleType(), True),
    StructField("Global_intensity", DoubleType(), True),
    StructField("Sub_metering_1", DoubleType(), True),
    StructField("Sub_metering_2", DoubleType(), True),
    StructField("Sub_metering_3", DoubleType(), True)
])

# 2. Parse JSON
parsed_stream_df = string_stream_df.withColumn("data", from_json(col("json_payload"), json_schema)) \
    .select("data.*")

# 3. Silver Layer Transformations (Date handling & Cleaning)
# Concatenate Date and Time, then cast to an actual Spark Timestamp
silver_transformed_df = parsed_stream_df.withColumn(
    "event_time",
    to_timestamp(concat_ws(" ", col("Date"), col("Time")), "yyyy-MM-dd HH:mm:ss")
)

# Strict filter and drop duplicates based on the new valid timestamp
silver_stream_df = silver_transformed_df \
    .filter(col("Global_active_power").isNotNull()) \
    .dropDuplicates(["meter_id", "event_time"])

# 4. Write to memory
silver_query = silver_stream_df \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("live_silver_table") \
    .start()

print("Silver Layer: Data parsed, dates correctly formatted to Timestamp, and Nulls removed...")

Silver Layer: Data parsed, dates correctly formatted to Timestamp, and Nulls removed...


# Golden layer

In [ ]:
from pyspark.sql.functions import col, window, avg

# Step 1: Build the Gold Layer Aggregations
# Using a 5-minute window and watermark for aggregations
gold_stream_df = silver_stream_df \
    .withWatermark("event_time", "5 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("meter_id")
    ) \
    .agg(
        avg("Global_active_power").alias("avg_active_power"),
        avg("Voltage").alias("avg_voltage")
    )

# Step 2: Define paths for saving directly to the specific project folder
# These paths point exactly to where the folders were moved yesterday
gold_output_path = "/content/drive/MyDrive/Project DEPI/SmartGrid_Gold_Data"
checkpoint_path = "/content/drive/MyDrive/Project DEPI/SmartGrid_Checkpoints"

# Step 3: Write the aggregated data to permanent Parquet files
gold_query = gold_stream_df \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", gold_output_path) \
    .option("checkpointLocation", checkpoint_path) \
    .start()

print("Gold Layer: Calculating 5-minute aggregations and saving to Project DEPI folder as Parquet files...")

Gold Layer: Calculating 5-minute aggregations and saving to Project DEPI folder as Parquet files...


# Show results

In [ ]:

gold_output_path = "/content/drive/MyDrive/Project DEPI/SmartGrid_Gold_Data"

try:
    # Reading permanent Parquet files
    saved_gold_df = spark.read.parquet(gold_output_path)

   # Display data sorted from newest to oldest
    saved_gold_df.orderBy("window.start", ascending=False).show(truncate=False)
    print("Success: Reading permanent Parquet files directly from Google Drive!")

except Exception as e:
    print("The files are not yet complete! Spark is still collecting time window data. Please wait a few more minutes.")

+------------------------------------------+--------+-------------------+------------------+
|window                                    |meter_id|avg_active_power   |avg_voltage       |
+------------------------------------------+--------+-------------------+------------------+
|{2024-01-30 23:45:00, 2024-01-30 23:50:00}|house_1 |0.39708271446379845|229.58543076202736|
|{2024-01-30 23:40:00, 2024-01-30 23:45:00}|house_1 |0.3424169248829977 |NaN               |
|{2024-01-30 23:35:00, 2024-01-30 23:40:00}|house_1 |NaN                |229.0685413443617 |
|{2024-01-30 23:30:00, 2024-01-30 23:35:00}|house_1 |0.22064199640665333|230.51561481096542|
|{2024-01-30 23:25:00, 2024-01-30 23:30:00}|house_1 |0.4365581054596607 |230.44330666889658|
|{2024-01-30 23:20:00, 2024-01-30 23:25:00}|house_1 |0.28285996949912395|231.7927839970593 |
|{2024-01-30 23:15:00, 2024-01-30 23:20:00}|house_1 |0.2078812392549197 |NaN               |
|{2024-01-30 23:10:00, 2024-01-30 23:15:00}|house_1 |0.328918610075851

In [ ]:
# Stop all active streaming queries gracefully
for stream in spark.streams.active:
    stream.stop()

print("All Spark streams have been safely stopped.")

All Spark streams have been safely stopped.
